# AllSortsHub AI Video Studio (Colab, free-tier)

Turns a written episode script + AI-generated scene images into a narrated,
Ken-Burns-animated video (5-8+ minutes) with zero paid tools.

**Workflow**
1. Write your episode as JSON (`episode_ai_template.json` shows the format) — one entry per scene, each with narration text and an image filename.
2. Generate one image per scene for free using a web tool (Bing Image Creator, Leonardo.ai free credits, etc.) and put them in `colab/images/` in this repo (or upload directly in Colab).
3. Run the cells below in order. Free Microsoft neural TTS (`edge-tts`) generates narration, this notebook animates each still with a slow pan/zoom, burns in subtitles, mixes in optional background music, and stitches everything with ffmpeg — the same approach `studio.py` already uses for its final assembly step.
4. Download the final MP4 + vertical Shorts from `output/`.

No GPU required — this all runs on Colab's free CPU runtime.

## 1. Setup

In [ ]:
!pip -q install edge-tts moviepy pillow
!apt-get -y -qq install ffmpeg > /dev/null
print("Setup done.")

In [ ]:
import json, os, glob, shutil, subprocess, math, asyncio
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# Clone your repo so images/scripts/output all live in one place.
# Skip this cell (or point REPO elsewhere) if you already have the repo open in Colab.
REPO_URL = "https://github.com/parth01/AllSortsHub-Cartoon-Studio.git"
REPO = Path("/content/AllSortsHub-Cartoon-Studio")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)

IMAGES = REPO / "colab" / "images"       # put your AI-generated scene images here
MUSIC  = REPO / "colab" / "music"        # optional: one royalty-free .mp3 background track
BUILD  = Path("/content/build")
OUT    = REPO / "output"
for p in (IMAGES, MUSIC, BUILD, OUT):
    p.mkdir(parents=True, exist_ok=True)

W, H, FPS = 1920, 1080, 30
print("Repo ready at", REPO)

## 2. Write your episode script\n\nEdit `episode_ai_template.json` (in `colab/`) with your real narration and image filenames, or point `EP` at your own file. Keep narration to natural spoken sentences — that's what drives both timing and subtitles.

In [ ]:
EP = REPO / "colab" / "episode_ai_template.json"
print(Path(EP).read_text())

## 3. Helpers: fonts, subtitle wrapping, Ken Burns pan/zoom\n\nThe `wrap()` function mirrors the one already in `studio.py` so subtitle styling stays consistent with the rest of the project.

In [ ]:
def font(size, bold=False):
    paths = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
    ]
    for p in paths:
        if os.path.exists(p):
            return ImageFont.truetype(p, size)
    return ImageFont.load_default()

def wrap(d, text, f, maxw):
    lines = []; cur = ""
    for w in text.split():
        q = (cur + " " + w).strip()
        if d.textbbox((0, 0), q, font=f)[2] <= maxw:
            cur = q
        else:
            lines.append(cur); cur = w
    if cur:
        lines.append(cur)
    return lines

def load_cover(path, zoom=1.0):
    img = Image.open(path).convert("RGB")
    iw, ih = img.size
    target_ratio = W / H
    if iw / ih > target_ratio:
        new_h = int(H * zoom); new_w = int(new_h * iw / ih)
    else:
        new_w = int(W * zoom); new_h = int(new_w * ih / iw)
    return img.resize((new_w, new_h), Image.LANCZOS)

def ken_burns_frame(image_path, t, dur, subtitle="", pan="right", zoom_start=1.0, zoom_end=1.12):
    prog = min(1.0, t / max(dur, 0.01))
    zoom = zoom_start + (zoom_end - zoom_start) * prog
    big = load_cover(image_path, zoom)
    bw, bh = big.size
    max_x, max_y = bw - W, bh - H
    if pan == "right":
        x, y = int(max_x * prog), max_y // 2
    elif pan == "left":
        x, y = int(max_x * (1 - prog)), max_y // 2
    elif pan == "down":
        x, y = max_x // 2, int(max_y * prog)
    else:
        x, y = max_x // 2, int(max_y * (1 - prog))
    x = max(0, min(max_x, x)); y = max(0, min(max_y, y))
    frame = big.crop((x, y, x + W, y + H)).copy()
    d = ImageDraw.Draw(frame)
    if subtitle:
        f = font(44, True)
        lines = wrap(d, subtitle, f, 1500)
        boxh = len(lines) * 60 + 35
        ypos = H - 80 - boxh
        overlay = Image.new("RGBA", frame.size, (0, 0, 0, 0))
        od = ImageDraw.Draw(overlay)
        od.rounded_rectangle((170, ypos, 1750, H - 45), 30, fill=(0, 0, 0, 165))
        frame = Image.alpha_composite(frame.convert("RGBA"), overlay).convert("RGB")
        d = ImageDraw.Draw(frame)
        for line in lines:
            bb = d.textbbox((0, 0), line, font=f)
            d.text(((W - (bb[2] - bb[0])) / 2, ypos + 18), line, font=f, fill="white")
            ypos += 60
    return frame

## 4. Free narration via edge-tts

In [ ]:
import edge_tts

async def _synth(text, voice, out_path):
    communicate = edge_tts.Communicate(text, voice)
    await communicate.save(str(out_path))

def tts(text, voice, out_path):
    asyncio.run(_synth(text, voice, out_path))

def audio_duration(path):
    result = subprocess.run(
        ["ffprobe", "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrapper=1:nokey=1", str(path)],
        capture_output=True, text=True,
    )
    return float(result.stdout.strip())

# Some good free neural voices to try: en-US-GuyNeural, en-US-AriaNeural,
# en-GB-RyanNeural, en-US-JennyNeural, en-AU-WilliamNeural

## 5. Build the episode: per-scene render + full assembly

In [ ]:
def build_episode(ep_path, voice_default="en-US-GuyNeural"):
    data = json.loads(Path(ep_path).read_text())
    scenes = data["scenes"]
    scene_audio, scene_video = [], []

    for i, s in enumerate(scenes, 1):
        text = s["narration"]
        voice = s.get("voice", voice_default)
        a_path = BUILD / f"a{i:02d}.mp3"
        tts(text, voice, a_path)
        dur = audio_duration(a_path) + 0.4  # small padding so audio never gets cut off
        scene_audio.append(a_path)

        img_path = IMAGES / s["image"]
        sub = s.get("subtitle", text)
        pan = s.get("pan", ["right", "left", "down", "up"][i % 4])

        scene_dir = BUILD / f"s{i:02d}"
        scene_dir.mkdir(exist_ok=True)
        n_frames = max(1, int(dur * FPS))
        for j in range(n_frames):
            frame = ken_burns_frame(img_path, j / FPS, dur, sub, pan)
            frame.save(scene_dir / f"f{j:05d}.jpg", quality=90)

        v_path = BUILD / f"v{i:02d}.mp4"
        subprocess.run(
            ["ffmpeg", "-y", "-framerate", str(FPS), "-i", str(scene_dir / "f%05d.jpg"),
             "-c:v", "libx264", "-pix_fmt", "yuv420p", "-preset", "veryfast", "-crf", "19", str(v_path)],
            check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
        )
        scene_video.append(v_path)
        print(f"Scene {i}/{len(scenes)} rendered ({dur:.1f}s)")

    vlist = BUILD / "v_list.txt"
    vlist.write_text("".join(f"file '{p.as_posix()}'\n" for p in scene_video))
    silent = BUILD / "silent.mp4"
    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(vlist), "-c", "copy", str(silent)], check=True)

    alist = BUILD / "a_list.txt"
    alist.write_text("".join(f"file '{p.as_posix()}'\n" for p in scene_audio))
    narration = BUILD / "narration.mp3"
    subprocess.run(["ffmpeg", "-y", "-f", "concat", "-safe", "0", "-i", str(alist), "-c", "copy", str(narration)], check=True)

    final_audio = narration
    music_files = list(MUSIC.glob("*.mp3"))
    if music_files:
        mixed = BUILD / "mixed.mp3"
        subprocess.run([
            "ffmpeg", "-y", "-i", str(narration), "-stream_loop", "-1", "-i", str(music_files[0]),
            "-filter_complex", "[1:a]volume=0.15[m];[0:a][m]amix=inputs=2:duration=first:dropout_transition=2[aout]",
            "-map", "[aout]", str(mixed),
        ], check=True)
        final_audio = mixed

    master = OUT / f"{data.get('title', 'episode')}.mp4"
    subprocess.run([
        "ffmpeg", "-y", "-i", str(silent), "-i", str(final_audio),
        "-c:v", "copy", "-c:a", "aac", "-shortest", "-movflags", "+faststart", str(master),
    ], check=True)
    print("Master episode created:", master)
    return master

## 6. Auto-crop vertical Shorts from the finished master\n\nPick timestamps after you've watched the master once — `highlights` is a list of `(start_seconds, duration_seconds)`.

In [ ]:
def make_shorts(master_path, highlights):
    for k, (start, dur) in enumerate(highlights, 1):
        out = OUT / f"{master_path.stem}_Short_{k:02d}.mp4"
        subprocess.run([
            "ffmpeg", "-y", "-ss", str(start), "-t", str(dur), "-i", str(master_path),
            "-vf", "scale=-2:1920,crop=1080:1920:(in_w-1080)/2:0,format=yuv420p",
            "-c:v", "libx264", "-preset", "veryfast", "-crf", "21", "-c:a", "aac", "-movflags", "+faststart", str(out),
        ], check=True)
    print("Shorts created in", OUT)

## 7. Run it

In [ ]:
master = build_episode(EP)
# Adjust these after previewing the master once:
make_shorts(master, [(10, 20), (60, 20), (120, 20)])

## 8. Get the code into your repo

Two ways, no git auth hassle needed:

- **Easiest:** download this notebook (`File > Download > .ipynb`) and `episode_ai_template.json`, then use GitHub's web "Add file → Upload files" on your repo page to drop them into `colab/`.
- **From Colab directly** (needs a GitHub personal access token):
```
!git config --global user.email "you@example.com"
!git config --global user.name "your-username"
%cd /content/AllSortsHub-Cartoon-Studio
!git add colab/
!git commit -m "Add AI image + TTS narration video pipeline"
!git push https://<YOUR_TOKEN>@github.com/parth01/AllSortsHub-Cartoon-Studio.git main
```

**Reminder for monetization:** when you upload the finished video, toggle "Altered or synthetic content" in YouTube Studio, since the scene art is AI-generated. Keep writing a genuinely different script each episode — that's what keeps this on the right side of YouTube's inauthentic-content policy.